In [ ]:
import pandas as pd
import numpy as np
import os
import uuid

# ==========================================
# CONFIGURATION & CONSTANTS
# ==========================================
OUTPUT_DIR = "nac_relational_dataset"
os.makedirs(OUTPUT_DIR, exist_ok=True)

POPULATION_TARGETS = {
    2025: 350000, 2030: 1050000, 2035: 1950000, 
    2040: 3100000, 2045: 4600000, 2050: 6500000
}

# Unique ID Tracking
USED_NATIONAL_IDS = set()
USED_REAL_ESTATE_IDS = set()

CAREERS = {
    'Civil Engineering': (120000, 700000), 'Accounting': (80000, 400000),
    'Medicine': (150000, 1200000), 'Software Engineering': (180000, 900000),
    'Architecture': (120000, 800000), 'Pharmacy': (90000, 450000),
    'Digital Marketing': (100000, 600000), 'Nursing': (80000, 300000),
    'Data Science': (200000, 1000000), 'Cybersecurity': (250000, 1100000),
    'Unemployed/Student': (0, 0), 'Retired': (40000, 240000)
}
MAJORS = [k for k in CAREERS.keys() if k not in ['Unemployed/Student', 'Retired']]
MAJOR_PROBS = [0.25, 0.20, 0.12, 0.10, 0.08, 0.08, 0.07, 0.05, 0.03, 0.02]

# Additional Categories
RESIDENCE_SITES = ['R1', 'R2', 'R3', 'R4', 'R5', 'R6', 'R7', 'R8']
SITE_PROBS = [0.05, 0.10, 0.25, 0.10, 0.10, 0.05, 0.25, 0.10] 
RESIDENCE_TYPES = ['Apartment', 'Villa', 'Twin House', 'Town House', 'Studio']
TRANSPORT_TYPES = ['Monorail', 'LRT', 'Fly Taxi', 'Electric Vehicle', 'Public Bus', 'Private Car', 'Bicycle/Walk']
HOSPITAL_SERVICES = ['None', 'Cardiovascular', 'Oncology', 'Neurology', 'Physical Therapy', 'Mental Health', 'Dentistry']
CHRONIC_DISEASES = ['None', 'Type 2 Diabetes', 'Hypertension', 'Obesity', 'Asthma']

# ==========================================
# CORE LOGIC HELPERS
# ==========================================

def generate_national_id(yob, gender):
    century = "2" if yob < 2000 else "3"
    year_str = str(yob)[-2:]
    month, day = str(np.random.randint(1,13)).zfill(2), str(np.random.randint(1,29)).zfill(2)
    gov = str(np.random.choice([1, 2, 21])).zfill(2)
    while True:
        seq = np.random.randint(100, 999)
        if (gender == 'Male' and seq % 2 == 0) or (gender == 'Female' and seq % 2 != 0): seq += 1
        nat_id = f"{century}{year_str}{month}{day}{gov}{seq}1"
        if nat_id not in USED_NATIONAL_IDS:
            USED_NATIONAL_IDS.add(nat_id)
            return nat_id

def sync_education_status(age, existing_major):
    """Crucial logic: preserves major once assigned at age 19."""
    if age < 4: return "None", "None", "None", "None", "None"
    elif age <= 18:
        grades = ["Kindergarten", "Primary", "Preparatory", "Secondary"]
        idx = min(3, max(0, (age - 4) // 4))
        return grades[idx], "Basic", "Public", "None", "None"
    elif 19 <= age <= 23:
        major = existing_major if existing_major != "None" else np.random.choice(MAJORS, p=MAJOR_PROBS)
        return "University Student", "Higher Ed", "None", major, "Public"
    else:
        return "Graduate/Post-Graduate", "Higher Ed", "None", existing_major, "None"

def calculate_skewed_income(career):
    min_sal, max_sal = CAREERS.get(career, (0, 0))
    if max_sal == 0: return 0
    return int(np.random.triangular(min_sal, min_sal + (max_sal - min_sal) * 0.2, max_sal))

def append_to_csv(df, filename):
    filepath = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(filepath, mode='a', index=False, header=not os.path.exists(filepath), encoding='utf-8-sig')

# ==========================================
# RELATIONAL EXPORT ENGINE
# ==========================================

def extract_and_save_relational_data(df, is_new_batch, year):
    if df.empty: return
    
    # Dimensions (Write only for new arrivals)
    if is_new_batch:
        append_to_csv(df[['National_ID', 'Gender', 'Year_of_Birth', 'Year_of_Death']].drop_duplicates(), 'Dim_Resident.csv')
        append_to_csv(df[['Real_Estate_ID', 'Residence_Site', 'Residence_Type', 'Household_Green_Space_sqm']].drop_duplicates(), 'Dim_Real_Estate.csv')

    # Fact Tables (Write snapshot for entire living population)
    snap = df.copy()
    snap['Report_Year'] = year
    snap['Record_ID'] = [str(uuid.uuid4()) for _ in range(len(snap))]

    append_to_csv(snap[['Record_ID', 'Report_Year', 'National_ID', 'Real_Estate_ID', 'Current_Age', 'Marital_Status', 'Career', 'Income_per_Year_EGP', 'Primary_Transport_Type']], 'Fact_SocioEconomic.csv')
    append_to_csv(snap[['Record_ID', 'Report_Year', 'National_ID', 'Edu_Grade', 'Edu_Type', 'School_Type', 'University_Major', 'University_Type']], 'Fact_Education.csv')
    append_to_csv(snap[['Record_ID', 'Report_Year', 'National_ID', 'Health_Percent', 'Chronic_Disease', 'Hospital_Service_Needed', 'BMI']], 'Fact_Health_Medical.csv')
    
    # Environment (Household Level)
    env = snap[['Real_Estate_ID', 'Report_Year', 'Has_Solar_System', 'Has_AC', 'Uses_Greywater_Recycling', 'Avg_Monthly_Electricity_KW', 'Carbon_Footprint_Index']].drop_duplicates(subset=['Real_Estate_ID'])
    env['Record_ID'] = [str(uuid.uuid4()) for _ in range(len(env))]
    append_to_csv(env, 'Fact_Household_Environment.csv')

# ==========================================
# MAIN SIMULATION ENGINE
# ==========================================

def generate_batch(year, num):
    """Generates consistent initial data for new residents."""
    data = []
    for _ in range(num):
        age = int(np.random.exponential(25)) if np.random.random() > 0.3 else int(np.random.uniform(0, 80))
        yob = year - age
        gender = np.random.choice(['Male', 'Female'])
        
        # Education and Career Initialization
        grade, etype, stype, major, utype = sync_education_status(age, "None")
        career = 'Unemployed/Student' if age < 22 else ('Retired' if age >= 65 else major)
        
        data.append({
            'National_ID': generate_national_id(yob, gender), 'Real_Estate_ID': str(uuid.uuid4())[:14],
            'Year_of_Birth': yob, 'Year_of_Death': yob + int(np.random.normal(78, 10)), 'Gender': gender, 'Current_Age': age,
            'Marital_Status': 'Married' if age > 25 else 'Single', 'Career': career, 'Income_per_Year_EGP': calculate_skewed_income(career),
            'Edu_Grade': grade, 'Edu_Type': etype, 'School_Type': stype, 'University_Major': major, 'University_Type': utype,
            'Residence_Site': np.random.choice(RESIDENCE_SITES), 'Residence_Type': np.random.choice(RESIDENCE_TYPES), 'Household_Green_Space_sqm': np.random.randint(0, 100),
            'Health_Percent': np.random.randint(60, 100), 'Chronic_Disease': 'None', 'Hospital_Service_Needed': 'None', 'BMI': 24.5,
            'Has_Solar_System': 0, 'Has_AC': 1, 'Uses_Greywater_Recycling': 0, 'Avg_Monthly_Electricity_KW': 450, 'Carbon_Footprint_Index': 1.2,
            'Primary_Transport_Type': np.random.choice(TRANSPORT_TYPES)
        })
    return pd.DataFrame(data)

def run_simulation():
    print("🚀 Starting NAC Simulation v3.0 (Relational Sync Optimized)...")
    living_df = pd.DataFrame()

    for year, target in POPULATION_TARGETS.items():
        print(f"--- Year {year} ---")
        if not living_df.empty:
            living_df['Current_Age'] += 5
            
            # Update Education Grades & Transitions
            for i, row in living_df.iterrows():
                grade, etype, stype, major, utype = sync_education_status(row['Current_Age'], row['University_Major'])
                living_df.at[i, 'Edu_Grade'] = grade
                living_df.at[i, 'University_Major'] = major
                
                # Relational Pivot: Student -> Worker
                if row['Current_Age'] >= 22 and row['Career'] == 'Unemployed/Student' and major != 'None':
                    living_df.at[i, 'Career'] = major
                    living_df.at[i, 'Income_per_Year_EGP'] = calculate_skewed_income(major)
                
                # Retirement
                if row['Current_Age'] >= 65 and row['Career'] != 'Retired':
                    living_df.at[i, 'Career'] = 'Retired'
                    living_df.at[i, 'Income_per_Year_EGP'] = calculate_skewed_income('Retired')

            living_df = living_df[living_df['Year_of_Death'] > year]
            extract_and_save_relational_data(living_df, is_new_batch=False, year=year)

        shortfall = target - len(living_df)
        if shortfall > 0:
            new_batch = generate_batch(year, shortfall)
            extract_and_save_relational_data(new_batch, is_new_batch=True, year=year)
            living_df = pd.concat([living_df, new_batch], ignore_index=True)

    print(f"✅ Simulation complete. Files saved in /{OUTPUT_DIR}")

if __name__ == "__main__":
    run_simulation()

🚀 Starting NAC Simulation v3.0 (Relational Sync Optimized)...
--- Year 2025 ---
--- Year 2030 ---
